In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import numpy as np

import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

In [ ]:
BD = ButterflyDataset(
    modalities={"coords": None}, data_dir=os.path.join(os.environ["DATA_DIR"]), use_aux_data="all"
)

In [ ]:
BCB = ButterflyCaptionBuilder(
    data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
    templates_fname="v3.json",
    concepts_fname="v2.json",
    seed=42,
)

# BCB = DummyCaptionBuilder(
#     data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
#     templates_fname="v2.json",
#     seed=42,
# )

BM = BaseDataModule(
    dataset=BD,
    batch_size=32,
    split_mode="from_file",
    saved_split_file_name=os.path.join(
        os.environ["DATA_DIR"], "s2bms", "splits/split_indices_s2bms_2024-08-14-1459.pth"
    ),
    caption_builder=BCB,
)

In [ ]:
# batch = next(iter(BM.train_dataloader()))
batch = next(iter(BM.val_dataloader()))
batch.keys()

In [ ]:
import json

concepts_path = os.path.join(os.environ["DATA_DIR"], "s2bms", "concept_captions/v2.json")

concepts = json.load(open(concepts_path))
concepts

In [ ]:
float(batch["aux"]["aux"][0, 0])

In [ ]:
ind_aux = np.where(np.array(BD.use_aux_data["aux"]) == "aux_corine_frac_412")[0]
# BD.mode
# ind_aux = 70

list_vals = []
for i in range(len(BD)):
    aux_data = BD[i]["aux"]["aux"][ind_aux]
    list_vals.append(aux_data)

list_vals
import numpy as np
import torch


def find_elbow_point(vals):
    """vals is a list of tensor values"""
    with torch.no_grad():
        vals = torch.tensor(vals).cpu().numpy()
        vals = vals[~np.isnan(vals)]  # remove NaN values

        vals = np.sort(vals)
        vals = vals[vals > vals[0]]
        x = np.arange(len(vals)) / len(vals)
        y = vals
        slope = (y[-1] - y[0]) / (x[-1] - x[0])  # diagonal from first to last point
        intercept = y[0] - slope * x[0]
        orthogonal_slope = -1 / slope

        intercepts_orthogonal = y - orthogonal_slope * x
        intersection_diagonal_orthogonal = (intercepts_orthogonal - intercept) / (
            slope - orthogonal_slope
        )
        distances = np.sqrt(
            (x - intersection_diagonal_orthogonal) ** 2 + (y - (slope * x + intercept)) ** 2
        )  # distance to diagonal
        elbow_index = np.argmax(distances)
        elbow_point = y[elbow_index]
        return elbow_point


theta_k = find_elbow_point(list_vals)
print(f"Elbow point (theta_k) for aux index {ind_aux}: {theta_k}")
import matplotlib.pyplot as plt

points = np.sort(torch.tensor(list_vals).cpu().numpy())
plt.figure(figsize=(8, 5))
plt.plot(np.sort(points), marker="o", linestyle="-", markersize=4)
## plot diagonal

plt.axhline(theta_k, color="red", linestyle="--", label=f"Elbow Point: {theta_k:.4f}")
plt.title("Sorted Values with Elbow Point")
plt.xlabel("Index")
plt.ylabel("Value")